In [1]:
from pathlib import Path
import pandas as pd


In [6]:
from neuroalign_preprocessing.utils.sessions import sanitize_session_id, sanitize_subject_code


linked_sessions = pd.read_csv("~/Downloads/linked_sessions.csv")
linked_sessions["subject_code"] = linked_sessions["SubjectCode"].apply(sanitize_subject_code)
linked_sessions["session_id"] = linked_sessions["ScanID"].apply(sanitize_session_id)


In [2]:
qsiparc_path = Path("/media/storage/yalab-dev/derivatives/qsiparc")
cat12parc_path = Path("/media/storage/yalab-dev/BIDS/derivatives/cat12_parcellated/cat12")

# CAT 12 loading

In [31]:
import json

ATLAS = "Schaefer2018N400n7Tian2020S3"
MASK = "gm"

CAT12_template = "sub-{subject_code}/ses-{session_id}/anat/atlas-{atlas_name}/sub-{subject_code}_ses-{session_id}_atlas-{atlas_name}_space-MNI152NLin2009cAsym_res-01_tissue-{tissue}{mask_str}_parc.{extension}"

for i, row in linked_sessions.iterrows():
    subject_code = row["subject_code"]
    session_id = row["session_id"]
    if MASK is not None:
        mask_str = f"_mask-{MASK}"
    else:
        mask_str = ""
    cat12_file = CAT12_template.format(
        subject_code=subject_code,
        session_id=session_id,
        atlas_name=ATLAS,
        tissue="GM",
        mask_str=mask_str,
        extension="tsv",
    )
    cat12_file_path = cat12parc_path / cat12_file
    if not cat12_file_path.exists():
        continue
    cat12_metadata_file = cat12_file_path.with_suffix(".json")
    with open(cat12_metadata_file, "r") as f:
        cat12_metadata = json.load(f)
    session_df = pd.read_csv(cat12_file_path, sep="\t")
    # update with metadata
    for key, value in cat12_metadata.items():
        if isinstance(value, dict):
            # flatten dict
            for subkey, subvalue in value.items():
                session_df[f"{key}_{subkey}"] = subvalue
        else:
            session_df[key] = value
    break
display(session_df.head())
    

,index,label,network_label,label_7network,index_17network,label_17network,network_label_17network,atlas_name,network_id,volume_mm3,...,original_file,mask,parcellation_scheme_name,parcellation_scheme_image,parcellation_scheme_lut,space,resampling_target,background_label,software_version,timestamp
0,1,LH_Vis_1,Vis,7Networks_LH_Vis_1,61.0,17Networks_LH_DorsAttnA_TempOcc_2,DorsAttnA,Schaefer2018N400n7,NaN,1759.778416,...,/media/storage/yalab-dev/BIDS/derivatives/CAT1...,gm,Schaefer2018N400n7Tian2020S3,/media/storage/yalab-dev/voxelops/Schaefer2018...,/media/storage/yalab-dev/voxelops/Schaefer2018...,MNI152NLin2009cAsym,data,0,0.1.2,2026-02-15T10:38:07.563743+00:00
1,2,LH_Vis_2,Vis,7Networks_LH_Vis_2,193.0,17Networks_LH_DefaultC_PHC_2,DefaultC,Schaefer2018N400n7,NaN,1877.039345,...,/media/storage/yalab-dev/BIDS/derivatives/CAT1...,gm,Schaefer2018N400n7Tian2020S3,/media/storage/yalab-dev/voxelops/Schaefer2018...,/media/storage/yalab-dev/voxelops/Schaefer2018...,MNI152NLin2009cAsym,data,0,0.1.2,2026-02-15T10:38:07.563743+00:00
2,3,LH_Vis_3,Vis,7Networks_LH_Vis_3,1.0,17Networks_LH_VisCent_ExStr_1,VisCent,Schaefer2018N400n7,NaN,1723.945662,...,/media/storage/yalab-dev/BIDS/derivatives/CAT1...,gm,Schaefer2018N400n7Tian2020S3,/media/storage/yalab-dev/voxelops/Schaefer2018...,/media/storage/yalab-dev/voxelops/Schaefer2018...,MNI152NLin2009cAsym,data,0,0.1.2,2026-02-15T10:38:07.563743+00:00
3,4,LH_Vis_4,Vis,7Networks_LH_Vis_4,13.0,17Networks_LH_VisPeri_ExStrInf_1,VisPeri,Schaefer2018N400n7,NaN,1616.086106,...,/media/storage/yalab-dev/BIDS/derivatives/CAT1...,gm,Schaefer2018N400n7Tian2020S3,/media/storage/yalab-dev/voxelops/Schaefer2018...,/media/storage/yalab-dev/voxelops/Schaefer2018...,MNI152NLin2009cAsym,data,0,0.1.2,2026-02-15T10:38:07.563743+00:00
4,5,LH_Vis_5,Vis,7Networks_LH_Vis_5,2.0,17Networks_LH_VisCent_ExStr_2,VisCent,Schaefer2018N400n7,NaN,1616.806330,...,/media/storage/yalab-dev/BIDS/derivatives/CAT1...,gm,Schaefer2018N400n7Tian2020S3,/media/storage/yalab-dev/voxelops/Schaefer2018...,/media/storage/yalab-dev/voxelops/Schaefer2018...,MNI152NLin2009cAsym,data,0,0.1.2,2026-02-15T10:38:07.563743+00:00


# QSIParc loading

In [ ]:
ATLAS = "Schaefer2018N400n7Tian2020S3"
MASK = "gm"

CAT12_template = "qsirecon-{workflow}/sub-{subject_code}/ses-{session_id}/dwi/atlas-{atlas_name}/sub-{subject_code}_ses-{session_id}_atlas-{atlas_name}_space-MNI152NLin2009cAsym_res-01_model-*_param-*_mask-{mask}_parc.tsv"

def _parse_entities(filename: str) -> dict[str, str]:
    """A simplified BIDS-like entity parser.

    Parameters
    ----------
    filename : str
        The filename to parse.

    Returns
    -------
    dict[str, str]
        A dictionary of BIDS entities.
    """
    entities = {}
    # Remove extensions like .nii.gz or .nii
    base_filename = filename.removesuffix(".nii.gz").removesuffix(".nii")
    parts = base_filename.split("_")

    for part in parts:
        if "-" in part:
            key, value = part.split("-", 1)
            entities[key] = value
    return {key: entities.get(key, None) for key in ["model","param"] if key in entities}


for i, row in linked_sessions.iterrows():
    subject_code = row["subject_code"]
    session_id = row["session_id"]
    for workflow in qsiparc_path.glob("qsirecon-*"):
        print(f"Checking workflow: {workflow.name}")
        workflow_name = workflow.name.split("-")[1]
        qsiparc_pattern = CAT12_template.format(
            workflow=workflow_name,
            subject_code=subject_code,
            session_id=session_id,
            atlas_name=ATLAS,
            mask=MASK,
        )
        qsiparc_files = list(qsiparc_path.glob(qsiparc_pattern))
        if len(qsiparc_files) == 0:
            continue
        for qsiparc_file in qsiparc_files:
            qsiparc_metadata_file = qsiparc_file.with_suffix(".json")
            with open(qsiparc_metadata_file, "r") as f:
                qsiparc_metadata = json.load(f)
            # update metadata with parsed entities from filename
            parsed_entities = _parse_entities(qsiparc_file.name)
            qsiparc_metadata.update(parsed_entities)
            qsiparc_df = pd.read_csv(qsiparc_file, sep="\t")
            # update with metadata
            for key, value in qsiparc_metadata.items():
                if isinstance(value, dict):
                    # flatten dict
                    for subkey, subvalue in value.items():
                        qsiparc_df[f"{key}_{subkey}"] = subvalue
                else:
                    qsiparc_df[key] = value
            display(qsiparc_df.head())
            break
        break
    break
        


Checking workflow: qsirecon-DIPYMAPMRI


,index,label,network_label,label_7network,index_17network,label_17network,network_label_17network,atlas_name,network_id,volume_mm3,...,parcellation_scheme_name,parcellation_scheme_image,parcellation_scheme_lut,space,resampling_target,background_label,software_version,timestamp,model,param
0,1,LH_Vis_1,Vis,7Networks_LH_Vis_1,61.0,17Networks_LH_DorsAttnA_TempOcc_2,DorsAttnA,Schaefer2018N400n7,NaN,-3726.494762,...,Schaefer2018N400n7Tian2020S3,/media/storage/yalab-dev/voxelops/Schaefer2018...,/media/storage/yalab-dev/voxelops/Schaefer2018...,MNI152NLin2009cAsym,data,0,0.1.2,2026-02-15T11:06:04.851604+00:00,mapmri,msd
1,2,LH_Vis_2,Vis,7Networks_LH_Vis_2,193.0,17Networks_LH_DefaultC_PHC_2,DefaultC,Schaefer2018N400n7,NaN,-8745.321034,...,Schaefer2018N400n7Tian2020S3,/media/storage/yalab-dev/voxelops/Schaefer2018...,/media/storage/yalab-dev/voxelops/Schaefer2018...,MNI152NLin2009cAsym,data,0,0.1.2,2026-02-15T11:06:04.851604+00:00,mapmri,msd
2,3,LH_Vis_3,Vis,7Networks_LH_Vis_3,1.0,17Networks_LH_VisCent_ExStr_1,VisCent,Schaefer2018N400n7,NaN,-7672.103566,...,Schaefer2018N400n7Tian2020S3,/media/storage/yalab-dev/voxelops/Schaefer2018...,/media/storage/yalab-dev/voxelops/Schaefer2018...,MNI152NLin2009cAsym,data,0,0.1.2,2026-02-15T11:06:04.851604+00:00,mapmri,msd
3,4,LH_Vis_4,Vis,7Networks_LH_Vis_4,13.0,17Networks_LH_VisPeri_ExStrInf_1,VisPeri,Schaefer2018N400n7,NaN,-6123.397246,...,Schaefer2018N400n7Tian2020S3,/media/storage/yalab-dev/voxelops/Schaefer2018...,/media/storage/yalab-dev/voxelops/Schaefer2018...,MNI152NLin2009cAsym,data,0,0.1.2,2026-02-15T11:06:04.851604+00:00,mapmri,msd
4,5,LH_Vis_5,Vis,7Networks_LH_Vis_5,2.0,17Networks_LH_VisCent_ExStr_2,VisCent,Schaefer2018N400n7,NaN,-10572.920508,...,Schaefer2018N400n7Tian2020S3,/media/storage/yalab-dev/voxelops/Schaefer2018...,/media/storage/yalab-dev/voxelops/Schaefer2018...,MNI152NLin2009cAsym,data,0,0.1.2,2026-02-15T11:06:04.851604+00:00,mapmri,msd
